<a href="https://colab.research.google.com/github/Saptaparnineogi/Customer-Churn-Prediction/blob/main/Telcom_Customer_Churn_Prediction_Model_building.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [96]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from google.colab import files

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier


from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
from sklearn import metrics
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
from sklearn.metrics import classification_report


In [2]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 9.5 MB/s eta 0:00:00


## Data Loading

In [4]:
uploadedfiles = files.upload()

Saving Churn.csv to Churn.csv


In [87]:
churn_df = pd.read_csv("Churn.csv")
churn_df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Data Processing

In [92]:
# Dropping customer id as it has no use
churn_df.drop("customerID", axis="columns", inplace=True)
# Relacing nu11 values with Median
churn_df['TotalCharges'] = pd.to_numeric(churn_df.TotalCharges, errors='coerce')
churn_df['TotalCharges'].fillna(churn_df['TotalCharges'].median(), inplace=True)
print(churn_df.shape)

(7043, 20)


/tmp/ipython-input-1641883875.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  churn_df['TotalCharges'].fillna(churn_df['TotalCharges'].median(), inplace=True)


In [93]:
churn_df['Churn'] = churn_df['Churn'].map({'Yes':1, 'No':0})

## Feature Engineering

In [29]:
# # tenure related
# churn_df['short_tenure'] = (churn_df['tenure'] < 6).astype(int)
# churn_df['mid_tenure'] = ((churn_df['tenure']>=6)&(churn_df['tenure']<24)).astype(int)

# # financial stress
# churn_df['high_charge'] = (churn_df['MonthlyCharges'] > 80).astype(int)
# churn_df['charge_per_month'] = churn_df['TotalCharges'] / (churn_df['tenure'] + 1)

# # contract risk
# churn_df['month_to_month'] = (churn_df['Contract']=='Month-to-month').astype(int)

# # service dissatisfaction
# churn_df['no_protection'] = (
#     (churn_df['OnlineSecurity']=='No') &
#     (churn_df['TechSupport']=='No')
# ).astype(int)

# # interaction
# churn_df['tenure_x_charge'] = churn_df['tenure'] * churn_df['MonthlyCharges']

## Train-Test Split

In [94]:
X = churn_df.drop('Churn', axis=1)
y = churn_df['Churn']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.3,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=42
)

## Baseline Model: Logistic regression

In [95]:
num_cols = X_train.select_dtypes(exclude='object').columns
cat_cols = X_train.select_dtypes(include='object').columns

preprocess = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first'), cat_cols)
])

lr = Pipeline([
    ('prep', preprocess),
    ('model', LogisticRegression(class_weight='balanced'))
])

lr.fit(X_train, y_train)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')),
                                                 ('cat',
                                                  OneHotEncoder(drop='first'),
                                                  Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object'))])),
                ('model', LogisticRegression(class_weight='balanced'))])

In [69]:
y_pred = lr.predict(X_test)
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.90      0.74      0.81       776
           1       0.52      0.78      0.62       281

    accuracy                           0.75      1057
   macro avg       0.71      0.76      0.72      1057
weighted avg       0.80      0.75      0.76      1057



#### Summary


* Catches most churners (78%)
* Acceptable false alarms (52% precision)
* Balanced overall performance

In [76]:
y_prob = lr.predict_proba(X_test)[:,1]
p, r, t = precision_recall_curve(y_test, y_prob)
df_pr = pd.DataFrame({
    'precision': p[:-1],
    'recall': r[:-1],
    'threshold': t
})

df_pr['f1'] = 2*(df_pr.precision*df_pr.recall) / \
             (df_pr.precision+df_pr.recall)

df_pr.sort_values('f1', ascending=False).head(10)

best_t = df_pr.sort_values('f1', ascending=False)\
              .iloc[0]['threshold']

y_pred_tuned = (y_prob > best_t).astype(int)

print(classification_report(y_test, y_pred_tuned))

              precision    recall  f1-score   support

           0       0.89      0.80      0.84       776
           1       0.57      0.72      0.63       281

    accuracy                           0.78      1057
   macro avg       0.73      0.76      0.74      1057
weighted avg       0.80      0.78      0.79      1057



* After threshold tuning we see slight improvement of F1 score for both the classes.
* Performance of LR is already very balanced

## Random Forest

In [81]:
from sklearn.ensemble import RandomForestClassifier

rf = Pipeline([
    ('prep', preprocess),
    ('model', RandomForestClassifier(
        n_estimators=300,
        class_weight='balanced',
        max_depth=10
    ))
])

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.89      0.80      0.84       776
           1       0.56      0.72      0.63       281

    accuracy                           0.78      1057
   macro avg       0.73      0.76      0.74      1057
weighted avg       0.80      0.78      0.79      1057



In [82]:
y_prob = rf.predict_proba(X_test)[:,1]

p, r, t = precision_recall_curve(y_test, y_prob)

df_pr = pd.DataFrame({
    'precision': p[:-1],
    'recall': r[:-1],
    'threshold': t
})

df_pr['f1'] = 2*(df_pr.precision*df_pr.recall) / \
             (df_pr.precision+df_pr.recall)

best_t = df_pr.sort_values('f1', ascending=False)\
              .iloc[0]['threshold']

y_pred = (y_prob > best_t).astype(int)

print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.89      0.81      0.84       776
           1       0.57      0.72      0.64       281

    accuracy                           0.78      1057
   macro avg       0.73      0.76      0.74      1057
weighted avg       0.80      0.78      0.79      1057



In [71]:
from xgboost import XGBClassifier

scale = (y_train==0).sum()/(y_train==1).sum()

xgb = Pipeline([
    ('prep', preprocess),
    ('model', XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        scale_pos_weight=scale,
        eval_metric='aucpr'
    ))
])

xgb.fit(X_train, y_train)
y_pred = xgb.predict(X_test)

print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.89      0.77      0.83       776
           1       0.54      0.74      0.63       281

    accuracy                           0.76      1057
   macro avg       0.72      0.76      0.73      1057
weighted avg       0.80      0.76      0.77      1057



In [83]:
y_prob = xgb.predict_proba(X_test)[:,1]

p, r, t = precision_recall_curve(y_test, y_prob)

df_pr = pd.DataFrame({
    'precision': p[:-1],
    'recall': r[:-1],
    'threshold': t
})

df_pr['f1'] = 2*(df_pr.precision*df_pr.recall) / \
             (df_pr.precision+df_pr.recall)

best_t = df_pr.sort_values('f1', ascending=False)\
              .iloc[0]['threshold']

y_pred = (y_prob > best_t).astype(int)

print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.90      0.76      0.82       776
           1       0.54      0.76      0.63       281

    accuracy                           0.76      1057
   macro avg       0.72      0.76      0.73      1057
weighted avg       0.80      0.76      0.77      1057



In [84]:
cat_cols = X_train.select_dtypes(include='object').columns.tolist()

model = CatBoostClassifier(
    iterations=500,
    eval_metric='PRAUC',
    class_weights=[1, scale]
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_cols,
    verbose=100
)


Learning rate set to 0.038444
0:	learn: 0.8009896	total: 18.1ms	remaining: 9.01s
100:	learn: 0.8511580	total: 2.35s	remaining: 9.27s
200:	learn: 0.8613287	total: 4.96s	remaining: 7.37s
300:	learn: 0.8765355	total: 6.93s	remaining: 4.58s
400:	learn: 0.8912744	total: 8.74s	remaining: 2.16s
499:	learn: 0.9028358	total: 10.5s	remaining: 0us


In [74]:
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.88      0.77      0.82       776
           1       0.53      0.71      0.61       281

    accuracy                           0.76      1057
   macro avg       0.71      0.74      0.72      1057
weighted avg       0.79      0.76      0.77      1057



In [85]:
y_prob = model.predict_proba(X_test)[:,1]

p, r, t = precision_recall_curve(y_test, y_prob)

df_pr = pd.DataFrame({
    'precision': p[:-1],
    'recall': r[:-1],
    'threshold': t
})

df_pr['f1'] = 2*(df_pr.precision*df_pr.recall) / \
             (df_pr.precision+df_pr.recall)

best_t = df_pr.sort_values('f1', ascending=False)\
              .iloc[0]['threshold']

y_pred = (y_prob > best_t).astype(int)

print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.90      0.75      0.82       776
           1       0.53      0.77      0.63       281

    accuracy                           0.76      1057
   macro avg       0.71      0.76      0.72      1057
weighted avg       0.80      0.76      0.77      1057



In [65]:
# churn_df.drop("TotalCharges", axis="columns", inplace=True)

#### Creation of tenure based feature `is_new`

In [66]:
churn_df["is_new"] = churn_df["tenure"].apply(lambda x: 1 if x < 6 else 0)

#### Feature scaling for Logistic regression model

In [75]:
preprocessor = ColumnTransformer(
    transformers=[
    ('num', StandardScaler(), numeric_cols),
    ('onehot', OneHotEncoder(handle_unknown='ignore'), onehot_cols)
    ] )

## Model Selection

In [78]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced'),
    "Random Forest": RandomForestClassifier(class_weight='balanced'),
    "XGBoost": XGBClassifier(eval_metric='logloss', scale_pos_weight = 2.76),
    "LightGBM": LGBMClassifier(class_weight='balanced',force_row_wise= True),
    "CatBoost": CatBoostClassifier(    loss_function='Logloss',
    eval_metric='PRAUC',
    class_weights=[1, 2.76],
    depth=6,
    learning_rate=0.05,
    iterations=1000)
}
pipelines = {
    name: Pipeline(steps=[('preprocess', preprocessor),
                         ('model', model)])
    for name, model in models.items()
}
from sklearn.metrics import classification_report
results = []
for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    probs = pipe.predict_proba(X_test)[:,1]
    print("----------------------------------------------")
    print(f"Model :{name}")
    print(classification_report(y_test, preds))
    # results.append({
    #     'Model': name,
    #     'ROC-AUC': roc_auc_score(y_test, probs),
    #     'F1': f1_score(y_test, preds),
    #     'Precision': precision_score(y_test, preds, pos_label=0),
    #     'Recall': recall_score(y_test, preds, pos_label=0)
    # })


# results_df = pd.DataFrame(results).sort_values(by='ROC-AUC', ascending=False)
# results_df

----------------------------------------------
Model :Logistic Regression
              precision    recall  f1-score   support

          No       0.91      0.69      0.78      1033
         Yes       0.48      0.80      0.60       374

    accuracy                           0.72      1407
   macro avg       0.69      0.74      0.69      1407
weighted avg       0.79      0.72      0.73      1407

----------------------------------------------
Model :Random Forest
              precision    recall  f1-score   support

          No       0.82      0.85      0.84      1033
         Yes       0.54      0.49      0.52       374

    accuracy                           0.75      1407
   macro avg       0.68      0.67      0.68      1407
weighted avg       0.75      0.75      0.75      1407



ValueError: Invalid classes inferred from unique values of `y`.  Expected: [0 1], got ['No' 'Yes']

## Observation:
1st iteration Result interpretation: On the first run for class 1 precision and recall both are quite low for all the models. Most probable cause for this is class imbalance issue.
2nd Iteration result interpretation: Updating the models so that it can deal with class imbalance improve recall for Logisic regressor, XGBoost and LightGBM. How ever we don't see much improvement for Random forest model wvwn while using balance class.

In [93]:
from catboost import CatBoostClassifier

X = churn_df.drop('Churn', axis=1)
y = churn_df['Churn']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)
cat_features = X.select_dtypes('object').columns.tolist()

model = CatBoostClassifier(
    iterations=1000,
    depth=6,
    learning_rate=0.05,
    loss_function='Logloss',
    eval_metric='PRAUC',
    class_weights=[1, 3],  # adjust
    verbose=0
)

model.fit(
    X_train, y_train,
    cat_features=cat_features,
    eval_set=(X_val, y_val)
)
eval

<function eval(source, globals=None, locals=None, /)>

In [97]:
from sklearn.metrics import classification_report, precision_score, recall_score

y_prob = model.predict_proba(X_test)[:,1]
# y_pred = model.predict(X_test)
y_pred = (y_prob > 0.5).astype(int)

print(classification_report(y_test, y_pred))

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

print("Precision:", precision)
print("Recall:", recall)


              precision    recall  f1-score   support

           0       0.91      0.71      0.80       775
           1       0.50      0.80      0.62       280

    accuracy                           0.74      1055
   macro avg       0.70      0.76      0.71      1055
weighted avg       0.80      0.74      0.75      1055

Precision: 0.5022522522522522
Recall: 0.7964285714285714


In [85]:
y_prob


array([0.35171679, 0.50606379, 0.08400357, ..., 0.17316699, 0.57743214,
       0.40690353])